# Capítulo 7 — Projeto de Banco de dados multi-tabelas: Povoando sua tabela

**Livro:** começa na página **263** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")
from banco import preparar
motor = preparar(7, "gabarito", recomecar=True)  # preparar(..., recomecar=True) volta o capítulo ao início
%reload_ext sql
%config SqlMagic.displaylimit = 60
%config SqlMagic.feedback = 0
%sql motor

## Encontrando uma namorada para Nigel · PDF p. 264

#### Exercício: Busca difícil numa tabela só

Encontre mulheres solteiras que gostam de `cinema` e de `trilhas`. Com tudo numa tabela só, a busca depende de `LIKE` em texto livre.

In [ ]:
%%sql
SELECT primeiro_nome, estado_civil, interesses FROM meus_contatos
WHERE sexo = 'F' AND estado_civil = 'solteira'
  AND interesses LIKE '%cinema%' AND interesses LIKE '%trilhas%';

## Pense além da tabela única · PDF p. 274

#### Exercício: Preparando a chave

Para ligar outras tabelas a `meus_contatos`, ela precisa de uma chave. Acrescente `contato_id SERIAL PRIMARY KEY`.

In [ ]:
%%sql
ALTER TABLE meus_contatos ADD COLUMN contato_id SERIAL PRIMARY KEY;
SELECT contato_id, primeiro_nome FROM meus_contatos ORDER BY contato_id;

## Restringindo sua chave externa · PDF p. 282

#### Exercício: A chave estrangeira protege

Tente inserir um interesse para o contato `999` (que não existe) com `try/except`.

In [ ]:
try:
    %sql INSERT INTO interesses (interesse, contato_id) VALUES ('pôquer', 999)
except Exception as erro:
    print("Bloqueado pela chave estrangeira:", "foreign key" in str(erro) or "chave estrangeira" in str(erro))

## CREATE uma tabela com uma CHAVE ESTRANGEIRA · PDF p. 283

#### Exercício: FOREIGN KEY

Crie `interesses` (`int_id SERIAL PRIMARY KEY`, `interesse`, `contato_id`) com uma `CONSTRAINT ... FOREIGN KEY (contato_id) REFERENCES meus_contatos (contato_id)`. Insira os interesses da contato 1 (um por linha).

In [ ]:
%%sql
CREATE TABLE interesses (
    int_id SERIAL PRIMARY KEY,
    interesse VARCHAR(50) NOT NULL,
    contato_id INT NOT NULL,
    CONSTRAINT meus_contatos_contato_id_fk FOREIGN KEY (contato_id) REFERENCES meus_contatos (contato_id)
);
INSERT INTO interesses (interesse, contato_id) VALUES ('cinema', 1), ('trilhas', 1), ('xadrez', 1);
SELECT * FROM interesses;

## Padrões de dados: um-a-um · PDF p. 285

#### Exercício: Um-para-um

Dados raramente usados podem ir para uma tabela à parte, com a **mesma** chave. Crie `contatos_extras` (`contato_id INT PRIMARY KEY REFERENCES meus_contatos`, `apelido`, `cor_favorita`) e insira 2 linhas.

In [ ]:
%%sql
CREATE TABLE contatos_extras (
    contato_id INT PRIMARY KEY REFERENCES meus_contatos (contato_id),
    apelido VARCHAR(20),
    cor_favorita VARCHAR(15)
);
INSERT INTO contatos_extras VALUES (1, 'Aninha', 'verde'), (2, 'Brunão', 'azul');
SELECT * FROM contatos_extras;

## Padrões de dados: um-para-muitos · PDF p. 286

#### Exercício: Um-para-muitos: profissões

Crie `profissoes` (`prof_id SERIAL PRIMARY KEY`, `profissao UNIQUE`) e preencha com as profissões distintas (`INSERT INTO ... SELECT DISTINCT`). Depois acrescente `prof_id` em `meus_contatos` com referência e preencha com `UPDATE ... FROM profissoes` (sintaxe do PostgreSQL).

In [ ]:
%%sql
CREATE TABLE profissoes (prof_id SERIAL PRIMARY KEY, profissao VARCHAR(50) UNIQUE NOT NULL);
INSERT INTO profissoes (profissao) SELECT DISTINCT profissao FROM meus_contatos ORDER BY profissao;
ALTER TABLE meus_contatos ADD COLUMN prof_id INT REFERENCES profissoes (prof_id);
UPDATE meus_contatos m SET prof_id = p.prof_id FROM profissoes p WHERE p.profissao = m.profissao;
SELECT primeiro_nome, profissao, prof_id FROM meus_contatos ORDER BY prof_id;

## Padrões de dados: nós precisamos de uma junction table · PDF p. 289

#### Exercício: Muitos-para-muitos com tabela de ligação

Palhaços fazem várias atividades e cada atividade é feita por vários palhaços. Crie `palhaco` (`palhaco_id SERIAL`, `nome UNIQUE`), `atividade` (`atividade_id SERIAL`, `atividade UNIQUE`) e a tabela de ligação `palhaco_atividade` com chave composta. Preencha `palhaco` a partir de `palhacos_info` e ligue alguns.

In [ ]:
%%sql
CREATE TABLE palhaco (palhaco_id SERIAL PRIMARY KEY, nome VARCHAR(30) UNIQUE NOT NULL);
CREATE TABLE atividade (atividade_id SERIAL PRIMARY KEY, atividade VARCHAR(30) UNIQUE NOT NULL);
CREATE TABLE palhaco_atividade (
    palhaco_id INT REFERENCES palhaco (palhaco_id),
    atividade_id INT REFERENCES atividade (atividade_id),
    PRIMARY KEY (palhaco_id, atividade_id)
);
INSERT INTO palhaco (nome) SELECT DISTINCT nome FROM palhacos_info ORDER BY nome;
INSERT INTO atividade (atividade) VALUES ('malabarismo'), ('balões'), ('mágica'), ('monociclo');
INSERT INTO palhaco_atividade
SELECT p.palhaco_id, a.atividade_id FROM palhaco p, atividade a
WHERE (p.nome = 'Pipoca' AND a.atividade IN ('malabarismo', 'balões'))
   OR (p.nome = 'Biribinha' AND a.atividade = 'mágica')
   OR (p.nome = 'Fuxico' AND a.atividade = 'malabarismo');
SELECT * FROM palhaco_atividade;

## Chaves compostas utilizam múltiplas colunas · PDF p. 294

#### Exercício: Conferindo a chave composta

Mostre as colunas que formam a chave primária de `palhaco_atividade` (`information_schema.key_column_usage`).

In [ ]:
%%sql
SELECT k.column_name, k.ordinal_position
FROM information_schema.table_constraints t
JOIN information_schema.key_column_usage k
  ON k.constraint_name = t.constraint_name AND k.table_schema = t.table_schema
WHERE t.table_schema = current_schema() AND t.table_name = 'palhaco_atividade'
  AND t.constraint_type = 'PRIMARY KEY'
ORDER BY k.ordinal_position;

## Segunda Forma Normal · PDF p. 301

#### Exercício: Tirando a dependência parcial (2NF)

Na tabela `estoque_lojas(brinquedo_id, loja_id, quantidade, endereco_loja)` a chave é (brinquedo_id, loja_id), mas `endereco_loja` depende só de `loja_id`. Crie a tabela ruim com 3 linhas e depois a versão 2NF: `lojas(loja_id, endereco)` e `estoque(brinquedo_id, loja_id, quantidade)`, preenchidas a partir dela.

In [ ]:
%%sql
CREATE TABLE estoque_lojas (brinquedo_id INT, loja_id INT, quantidade INT, endereco_loja VARCHAR(40),
                            PRIMARY KEY (brinquedo_id, loja_id));
INSERT INTO estoque_lojas VALUES (1, 10, 5, 'Rua A, 100'), (2, 10, 3, 'Rua A, 100'), (1, 20, 7, 'Av. B, 200');
CREATE TABLE lojas AS SELECT DISTINCT loja_id, endereco_loja AS endereco FROM estoque_lojas;
ALTER TABLE lojas ADD PRIMARY KEY (loja_id);
CREATE TABLE estoque AS SELECT brinquedo_id, loja_id, quantidade FROM estoque_lojas;
ALTER TABLE estoque ADD PRIMARY KEY (brinquedo_id, loja_id),
                    ADD FOREIGN KEY (loja_id) REFERENCES lojas (loja_id);
SELECT * FROM lojas;

## Terceira Forma Normal (finalmente) · PDF p. 306

#### Exercício: Tirando a dependência transitiva (3NF)

Em `cursos(curso_id, curso, professor, sala_do_professor)`, a sala depende do professor (que não é chave): dependência transitiva. Crie a tabela com 3 linhas e separe em `professores(professor, sala)` e `cursos_3nf(curso_id, curso, professor)`.

In [ ]:
%%sql
CREATE TABLE cursos (curso_id INT PRIMARY KEY, curso VARCHAR(30), professor VARCHAR(30), sala_do_professor VARCHAR(10));
INSERT INTO cursos VALUES (1, 'SQL Básico', 'Marta', 'B-12'), (2, 'SQL Avançado', 'Marta', 'B-12'),
                          (3, 'Python', 'Caio', 'C-03');
CREATE TABLE professores AS SELECT DISTINCT professor, sala_do_professor AS sala FROM cursos;
ALTER TABLE professores ADD PRIMARY KEY (professor);
CREATE TABLE cursos_3nf AS SELECT curso_id, curso, professor FROM cursos;
ALTER TABLE cursos_3nf ADD PRIMARY KEY (curso_id),
                       ADD FOREIGN KEY (professor) REFERENCES professores (professor);
SELECT * FROM professores;